# Create Stack Model

## Set Up

Import packages

In [1]:
import sys

sys.path.append("../")

from src.data_utils import get_data, get_models
from src.config import SEED, BASE_PATH
from src.nn_model import load_nn_clf
from src.feat_eng import calibrate_model
from src.tune import save_model, log

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score

import numpy as np
import pandas as pd

print(f"Path: {BASE_PATH}")

Import Models

In [2]:
OUTCOME_LIST = [
    "SERIOUS",
    "ANY",
    "PNEUMO",
    "CARDIAC_COMP",
    "VTE",
    "SEPSIS",
    "SSI",
    "UTI",
    "RENAL",
    "UNPLNREOP",
    "MORT",
]

In [3]:
df_dummy = pd.read_parquet(
    BASE_PATH / "data" / "processed" / OUTCOME_LIST[0] / "X_train.parquet"
)
X_shape = df_dummy.shape[1]
# Models
model_dir = BASE_PATH / "models" / "trained"
model_prefix_list = ["lgbm", "lr", "xgb"]
MODEL_DICT = {}

for outcome in OUTCOME_LIST:
    ## Base models
    MODEL_DICT[outcome] = get_models(model_prefix_list, outcome, model_dir)
    ## Neural network
    nn_import = load_nn_clf(
        data_path=BASE_PATH / "models" / "trained" / outcome / "nn.pt",
        in_dim=X_shape,
        device="cpu",
    )
    MODEL_DICT[outcome]["nn"] = nn_import

## Build Model

In [4]:
def build_stack(estimators, outcome, data_dir, n_splits, seed, n_jobs, save_dir):
    """
    Params
    -----
    estimators: list of (str, estimator)
    """
    outcome_dict = get_data(outcome_folder=outcome, file_dir=data_dir)
    ## Train
    X_train = outcome_dict["X_train"]
    y_train = outcome_dict["y_train"].values.ravel()
    X_val = outcome_dict["X_val"]
    y_val = outcome_dict["y_val"].values.ravel()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    # Ridge log regression
    meta_learner = LogisticRegression(
        random_state=seed,
        l1_ratio=0.0,
        solver="lbfgs",
        C=1.0,
        max_iter=1000,
    )
    log(f"Starting stack for {outcome}...")
    stack_model = StackingClassifier(
        estimators=estimators,
        final_estimator=meta_learner,
        cv=skf,
        stack_method="predict_proba",
        n_jobs=n_jobs,
        passthrough=False,  # only consider base estimators preds, not actual training feats
        verbose=1,
    )
    stack_model.fit(X_train, y_train)
    log("STACKED!...")
    weights = dict(
        zip(
            [name for name, _ in stack_model.estimators],
            stack_model.final_estimator_.coef_[0],
        )
    )
    log(weights)
    ## Calibrate model
    cal_model = calibrate_model(
        X=X_val,
        y=y_val,
        n_splits=n_splits,
        seed=seed,
        model=stack_model,
        n_cv_jobs=n_jobs,
    )
    log("CALIBRATED!")
    ## ensure performance doesn't degrade
    eval_dict = {
        "base_train": stack_model.predict_proba(X_train)[:, 1],
        "base_val": stack_model.predict_proba(X_val)[:, 1],
        "cal_train": cal_model.predict_proba(X_train)[:, 1],
        "cal_val": cal_model.predict_proba(X_val)[:, 1],
    }
    log("Evaluating...")
    for name, proba in eval_dict.items():
        if "val" in name:
            true = y_val
        elif "train" in name:
            true = y_train
        else:
            raise ValueError(name)
        ap_score = average_precision_score(y_true=true, y_score=proba)
        event_rate = float(np.mean(true))
        log(f"\t{name} score: {ap_score:.4f} ({(ap_score / event_rate):.1f})")
    log("Saving...")
    ## Save models
    base_save_dir = save_dir / "trained" / outcome
    cal_save_dir = save_dir / "calibrated" / outcome
    save_model(stack_model, "stack", save_dir=base_save_dir, calibrated=False)
    save_model(cal_model, "stack", save_dir=cal_save_dir, calibrated=True)
    log("DONE!")
    return stack_model, cal_model

Run sequentially

In [5]:
for outcome in OUTCOME_LIST:
    stack_model, cal_model = build_stack(
        estimators=list(MODEL_DICT[outcome].items()),
        outcome=outcome,
        data_dir=BASE_PATH / "data/processed",
        n_splits=3,
        seed=SEED,
        n_jobs=-1,
        save_dir=BASE_PATH / "models",
    )